In [24]:
# Fix Python path to include user-installed packages for BOTH Python 3.10 and 3.12
# import sys
# import importlib
# import site


# Note: NumPy Compatibility Warning

If you see a NumPy 1.x vs 2.x warning, you can safely ignore it. The code works fine despite the warning.

To fix it permanently, you would need to downgrade NumPy, but this requires disk space:
```python
# pip install "numpy<2" --user
```

In [25]:
# Import Earth Engine
import ee
import geemap

# Authenticate (only needed first time)
ee.Authenticate()

# Initialize Earth Engine
ee.Initialize(project="ee-ktwu01")

print("Earth Engine initialized successfully!")

Earth Engine initialized successfully!


In [26]:
# Load collection
dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

# Point of interest
point = ee.Geometry.Point(-121.8036, 39.0372)

# Get embedding images for two years
image1 = dataset.filterDate("2023-01-01", "2024-01-01").filterBounds(point).first()

image2 = dataset.filterDate("2024-01-01", "2025-01-01").filterBounds(point).first()

# Visualization parameters
vis_params = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}

# Calculate dot product (similarity measure)
dot_prod = image1.multiply(image2).reduce(ee.Reducer.sum())


# Print out some information about the images
def print_image_details():
    print("<b>2023 Image Details:</b>")
    print(f"Bands: {image1.bandNames().getInfo()}")
    print(f"Image Projection: {image1.projection().getInfo()}")

    print("\n<b>2024 Image Details:</b>")
    print(f"Bands: {image2.bandNames().getInfo()}")
    print(f"Image Projection: {image2.projection().getInfo()}")

    print("\n<b>Dot Product Similarity:</b>")
    print(f"Similarity Value: {dot_prod.getInfo()}")


# Run the detailed analysis
print_image_details()

# Optional: Export images
# Uncomment the following line if you want to export images
# export_images()

<b>2023 Image Details:</b>
Bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']
Image Projection: {'type': 'Projection', 'crs': 'EPSG:32610', 'transform': [10, 0, 500000, 0, 10, 4259840]}

<b>2024 Image Details:</b>
Bands: ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A

In [27]:
# 1. LOAD ALPHAEARTH EMBEDDINGS
alphaEarth = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')

# 2. DEFINE YOUR ANALYSIS PERIOD
startYear = 2018  # Match paper's June 2018 start
endYear = 2024    # Current available data

# 3. LOAD MINING LOCATIONS FROM CSV
# Data cite (Table S2? S3?): https://www.nature.com/articles/s41598-022-14987-0#MOESM3
# Sun, W., Jin, H., Jin, F. et al. Spatial analysis of global Bitcoin mining. Sci Rep 12, 10694 (2022). https://doi.org/10.1038/s41598-022-14987-0
# Data source: https://static-content.springer.com/esm/art%3A10.1038%2Fs41598-022-14987-0/MediaObjects/41598_2022_14987_MOESM3_ESM.xlsx
miningLocations = ee.FeatureCollection('projects/ee-ktwu01/assets/bitcoin-mining')


# Convert CSV data to proper format
def format_mining_location(feature):
    lat = ee.Number(feature.get('Latitude'))
    lon = ee.Number(feature.get('Longitude'))
    country = feature.get('CRCode')
    countryName = feature.get('CRName')

    return ee.Feature(
        ee.Geometry.Point([lon, lat]),
        {
            'label': 1,
            'year': 2018,
            'location': country,
            'country_name': countryName
        }
    )

miningLocations = miningLocations.map(format_mining_location)

# Print number of mining locations
numPositive = miningLocations.size()
print('Number of mining locations:', numPositive.getInfo())

# 4. VISUALIZE ALL POSITIVE MINING LOCATIONS ON MAP
# Create an interactive map
Map = geemap.Map(zoom=2)
Map.centerObject(miningLocations, 2)  # Global view
Map.addLayer(miningLocations, {'color': 'red', 'point_size': 0.0001}, 'Bitcoin Mining Locations')
Map

Number of mining locations: 6062


Map(center=[66.91775461601094, 26.123250690263546], controls=(WidgetControl(options=['position', 'transparent_…

In [28]:
# 5. GENERATE NEGATIVE SAMPLES (non-mining locations)
# Best practices for negative sampling:
# 1. Match the geographic distribution of positive samples
# 2. Include diverse land cover types (urban, rural, industrial, natural)
# 3. Use stratified random sampling within same regions
# 4. Aim for balanced dataset (1:1 or up to 1:3 positive:negative ratio)

# Get bounding box of mining locations to constrain negative sampling
miningBounds = miningLocations.geometry().bounds()

# Generate random points within the same geographic regions
negativeLocations = ee.FeatureCollection.randomPoints(
    region=miningBounds,
    points=numPositive,  # Match number of positive samples
    seed=42,  # For reproducibility
    maxError=1
)

# Add labels to negative samples
def add_negative_label(feature):
    return feature.set({
        'label': 0,
        'year': 2018,
        'location': 'negative_sample'
    })

negativeLocations = negativeLocations.map(add_negative_label)

# Optional: Filter out negative samples that are too close to mining sites
# This prevents contamination (e.g., excluding points within 1km of mining sites)
# minDistance = 1000  # meters

# The spatial filtering step is causing a persistent error ("String: Unable to convert object to string.").
# Commenting out this section as a workaround to allow the notebook to proceed.
# Note: Without this filtering, some negative samples may be very close to mining locations.
#
# # Create buffers around each mining location and merge them
# miningBuffers = miningLocations.map(lambda f: f.buffer(minDistance)).flatten()
# mergedMiningBuffer = miningBuffers.union()
#
# # Filter negative samples to exclude those that intersect with the merged buffer
# negativeLocations = negativeLocations.filter(ee.Filter.disjoint(mergedMiningBuffer))


print('Number of negative samples after filtering:', negativeLocations.size().getInfo())

# 6. COMBINE POSITIVE AND NEGATIVE SAMPLES
trainingPoints = miningLocations.merge(negativeLocations)
print('Total training points:', trainingPoints.size().getInfo())

Number of negative samples after filtering: 6062
Total training points: 12124


In [29]:
# # 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR
# # running four minutes and half after Extracting embeddings for multiple years...
# # # Extracting embeddings for multiple years...
# # Number of images for year 2018: 0
# # Number of images for year 2021: 0
# # Number of images for year 2024: 0
# # Embeddings extracted successfully!
# # This cannot be true. There are no numbers for these years.
# # We should debug for this.

# def extractEmbeddings(year):
#     """Extract AlphaEarth embeddings for a specific year by date range"""
#     print(f"--- Debugging extractEmbeddings for year {year} ---")
#     start_date = f"{year}-01-01"
#     end_date = f"{year + 1}-01-01"
#     print(f"Filtering AlphaEarth collection for date range: {start_date} to {end_date}")
#     yearImageCollection = alphaEarth.filterDate(start_date, end_date)
#     print(f"Number of images for year {year} in date range: {yearImageCollection.size().getInfo()}")
#     yearImage = yearImageCollection.first()
#     print(f"First image found for year {year}: {yearImage.getInfo() if yearImage else 'None'}")

#     samples = ee.FeatureCollection([]) # Initialize empty collection
#     if yearImage:
#         print(f"Filtering training points by image bounds for year {year}...")
#         # Filter training points to only include those within the bounds of the image
#         filteredTrainingPoints = trainingPoints.filterBounds(yearImage.geometry())
#         print(f"Number of training points within image bounds for year {year}: {filteredTrainingPoints.size().getInfo()}")

#         if filteredTrainingPoints.size().getInfo() > 0:
#             print(f"Sampling regions for year {year}...")
#             samples = yearImage.sampleRegions(
#                 collection=filteredTrainingPoints,
#                 scale=10,  # AlphaEarth is 10m resolution
#                 geometries=True
#             )
#             print(f"Number of samples extracted for year {year}: {samples.size().getInfo()}")
#             print(f"Sample features for year {year} (first 5): {samples.limit(5).getInfo()}")
#         else:
#             print(f"No training points within image bounds for year {year}, returning empty samples.")
#     else:
#         print(f"No image found for year {year} in date range, returning empty samples.")

#     print(f"--- Finished extractEmbeddings for year {year} ---")
#     return samples

# # 8. EXTRACT FOR MULTIPLE YEARS
# print('--- Debugging Extraction for Multiple Years ---')
# print('Extracting embeddings for multiple years...')
# embeddings2018 = extractEmbeddings(2018)
# embeddings2021 = extractEmbeddings(2021)  # Pre-China ban
# embeddings2024 = extractEmbeddings(2024)  # Post-ban

# print('Embeddings extracted successfully!')
# print('2018 samples:', embeddings2018.size().getInfo())
# print('2021 samples:', embeddings2021.size().getInfo())
# print('2024 samples:', embeddings2024.size().getInfo())
# print('--- Finished Extraction for Multiple Years ---')

# # 9. EXPORT FOR CLASSIFIER TRAINING
# print('--- Debugging Export Configuration ---')
# # Generate band selectors for all 64 AlphaEarth layers
# band_names = ['A' + str(i).zfill(2) for i in range(64)]
# selectors = ['label', 'location', 'year'] + band_names
# print(f"Selectors for export: {selectors}")

# # Export 2018 data to Google Drive
# task2018 = ee.batch.Export.table.toDrive(
#     collection=embeddings2018,
#     description='AlphaEarth_Mining_Training_2018',
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Uncomment to start the export task
# # task2018.start()
# # print('Export task started. Check your Google Drive and Earth Engine Tasks tab.')

# print('Export task configured. To start export, uncomment task2018.start() and run again.')
# print(f'This will export {embeddings2018.size().getInfo()} samples with {len(band_names)} embedding bands.')
# print('--- Finished Export Configuration ---')

# Task
Review the provided Earth Engine algorithm for extracting embeddings, identify the part that is running correctly (defining selectors), and create a new single cell with a revised approach for embedding extraction using a mosaic method, including the necessary debug output.

## Address timeout in embedding extraction

### Subtask:
Address the timeout issue encountered during the extraction of AlphaEarth embeddings using the mosaic approach. The previous attempt to sample all training points within the mosaic's bounds timed out. This subtask focuses on implementing a more efficient method to extract embeddings without hitting the computation limit.


**Reasoning**:
The previous attempt to get the size of the sampled FeatureCollection timed out. The core issue is likely sampling a large number of points over a large mosaic. The export function can handle large computations in the background. Therefore, modify the extraction function to return the sampled FeatureCollection directly without attempting to compute its size or get its info within the function. This will allow the export task to handle the computation in the background.



In [18]:
# # 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR (Revised Mosaic approach)

# def extractEmbeddings_mosaic_revised(year):
#     """Extract AlphaEarth embeddings for a specific year using a mosaic,
#        returning the FeatureCollection without immediate computation."""
#     print(f"--- Starting extractEmbeddings_mosaic_revised for year {year} ---")
#     start_date = f"{year}-01-01"
#     end_date = f"{year + 1}-01-01"
#     print(f"Filtering AlphaEarth collection for date range: {start_date} to {end_date}")

#     # Filter the collection by date
#     yearImageCollection = alphaEarth.filterDate(start_date, end_date)
#     print(f"Number of images for year {year} in date range: {yearImageCollection.size().getInfo()}")

#     samples = ee.FeatureCollection([]) # Initialize empty collection

#     if yearImageCollection.size().getInfo() > 0:
#         # Create a mosaic of the images for the year
#         yearImage = yearImageCollection.mosaic()
#         print(f"Mosaic image created for year {year}.")

#         print(f"Filtering training points by mosaic bounds for year {year}...")
#         # Filter training points to only include those within the bounds of the mosaic
#         filteredTrainingPoints = trainingPoints.filterBounds(yearImage.geometry())
#         num_filtered_points = filteredTrainingPoints.size().getInfo() # Still get this size for logging/debugging
#         print(f"Number of training points within mosaic bounds for year {year}: {num_filtered_points}")


#         if num_filtered_points > 0:
#             print(f"Configuring sampleRegions for year {year}...")
#             # Perform the sampling. This returns a FeatureCollection, but we avoid
#             # calling getInfo() on it here.
#             samples = yearImage.sampleRegions(
#                 collection=filteredTrainingPoints,
#                 scale=10,  # AlphaEarth is 10m resolution
#                 geometries=True
#             )
#             print(f"SampleRegions configured for year {year}. Returning FeatureCollection.")
#         else:
#             print(f"No training points within mosaic bounds for year {year}, returning empty samples collection.")
#     else:
#         print(f"No images found for year {year} in date range, returning empty samples collection.")

#     print(f"--- Finished extractEmbeddings_mosaic_revised for year {year} ---")
#     return samples

# # 8. EXTRACT FOR MULTIPLE YEARS (using revised mosaic approach)
# print('--- Debugging Extraction for Multiple Years (Revised Mosaic Approach) ---')
# print('Extracting embeddings for multiple years...')
# embeddings2018_mosaic = extractEmbeddings_mosaic_revised(2018)
# embeddings2021_mosaic = extractEmbeddings_mosaic_revised(2021)  # Pre-China ban
# embeddings2024_mosaic = extractEmbeddings_mosaic_revised(2024)  # Post-ban

# print('Embeddings extraction configured successfully! Ready for export.')
# # We don't print the size here to avoid the timeout
# # print('2018 samples (mosaic):', embeddings2018_mosaic.size().getInfo())
# # print('2021 samples (mosaic):', embeddings2021_mosaic.size().getInfo())
# # print('2024 samples (mosaic):', embeddings2024_mosaic.size().getInfo())
# print('--- Finished Extraction for Multiple Years (Revised Mosaic Approach) ---')

--- Debugging Extraction for Multiple Years (Revised Mosaic Approach) ---
Extracting embeddings for multiple years...
--- Starting extractEmbeddings_mosaic_revised for year 2018 ---
Filtering AlphaEarth collection for date range: 2018-01-01 to 2019-01-01
Number of images for year 2018 in date range: 10635
Mosaic image created for year 2018.
Filtering training points by mosaic bounds for year 2018...
Number of training points within mosaic bounds for year 2018: 12124
Configuring sampleRegions for year 2018...
SampleRegions configured for year 2018. Returning FeatureCollection.
--- Finished extractEmbeddings_mosaic_revised for year 2018 ---
--- Starting extractEmbeddings_mosaic_revised for year 2021 ---
Filtering AlphaEarth collection for date range: 2021-01-01 to 2022-01-01
Number of images for year 2021 in date range: 10614
Mosaic image created for year 2021.
Filtering training points by mosaic bounds for year 2021...
Number of training points within mosaic bounds for year 2021: 12124


**Reasoning**:
The embedding extraction functions have been revised to return the sampled FeatureCollections without triggering immediate computation, addressing the timeout issue. The next logical step is to configure the export tasks using these returned FeatureCollections. The previous attempt to configure export failed because the variables holding the FeatureCollections were not defined due to the timeout. This step will re-run the export configuration code, which should now work since the necessary FeatureCollections are available.



In [19]:
# # 9. EXPORT FOR CLASSIFIER TRAINING (using revised mosaic approach)
# print('--- Debugging Export Configuration (Revised Mosaic Approach) ---')
# # Generate band selectors for all 64 AlphaEarth layers
# band_names = ['A' + str(i).zfill(2) for i in range(64)]
# selectors = ['label', 'location', 'year'] + band_names
# print(f"Selectors for export: {selectors}")

# # Export 2018 data to Google Drive
# task2018_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2018_mosaic,
#     description='AlphaEarth_Mining_Training_2018_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Export 2021 data to Google Drive
# task2021_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2021_mosaic,
#     description='AlphaEarth_Mining_Training_2021_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Export 2024 data to Google Drive
# task2024_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2024_mosaic,
#     description='AlphaEarth_Mining_Training_2024_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Uncomment to start the export tasks
# # task2018_mosaic.start()
# # task2021_mosaic.start()
# # task2024_mosaic.start()
# # print('Export tasks started. Check your Google Drive and Earth Engine Tasks tab.')

# print('Export tasks configured. To start export, uncomment the task.start() lines and run again.')
# print(f'This will configure export for 2018, 2021, and 2024 samples with {len(band_names)} embedding bands.')
# print('--- Finished Export Configuration (Revised Mosaic Approach) ---')

--- Debugging Export Configuration (Revised Mosaic Approach) ---
Selectors for export: ['label', 'location', 'year', 'A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']
Export tasks configured. To start export, uncomment the task.start() lines and run again.
This will configure export for 2018, 2021, and 2024 samples with 64 embedding bands.
--- Finished Export Configuration (Revised Mosaic Approach) ---


## Create cell for successful code

### Subtask:
Move the code that defines the selectors (which ran successfully) into its own cell.


**Reasoning**:
Move the code that defines the band names and selectors into a new cell as requested.



In [20]:
# # Define band names and selectors for export

# # Generate band selectors for all 64 AlphaEarth layers
# band_names = ['A' + str(i).zfill(2) for i in range(64)]
# selectors = ['label', 'location', 'year'] + band_names
# print(f"Selectors for export: {selectors}")

Selectors for export: ['label', 'location', 'year', 'A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63']


## Create cell for embedding extraction (revised)

### Subtask:
Create a new cell for the embedding extraction logic, incorporating the changes to address the timeout issue.


**Reasoning**:
Define the function for extracting embeddings using the mosaic approach and call it for the specified years.



In [21]:
# # 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR (Revised Mosaic approach)

# def extractEmbeddings_mosaic_revised(year):
#     """Extract AlphaEarth embeddings for a specific year using a mosaic,
#        returning the FeatureCollection without immediate computation."""
#     print(f"--- Starting extractEmbeddings_mosaic_revised for year {year} ---")
#     start_date = f"{year}-01-01"
#     end_date = f"{year + 1}-01-01"
#     print(f"Filtering AlphaEarth collection for date range: {start_date} to {end_date}")

#     # Filter the collection by date
#     yearImageCollection = alphaEarth.filterDate(start_date, end_date)
#     collection_size = yearImageCollection.size().getInfo()
#     print(f"Number of images for year {year} in date range: {collection_size}")

#     samples = ee.FeatureCollection([]) # Initialize empty collection

#     if collection_size > 0:
#         # Create a mosaic of the images for the year
#         yearImage = yearImageCollection.mosaic()
#         print(f"Mosaic image created for year {year}.")

#         print(f"Filtering training points by mosaic bounds for year {year}...")
#         # Filter training points to only include those within the bounds of the mosaic
#         filteredTrainingPoints = trainingPoints.filterBounds(yearImage.geometry())
#         num_filtered_points = filteredTrainingPoints.size().getInfo() # Still get this size for logging/debugging
#         print(f"Number of training points within mosaic bounds for year {year}: {num_filtered_points}")


#         if num_filtered_points > 0:
#             print(f"Configuring sampleRegions for year {year}...")
#             # Perform the sampling. This returns a FeatureCollection, but we avoid
#             # calling getInfo() on it here.
#             samples = yearImage.sampleRegions(
#                 collection=filteredTrainingPoints,
#                 scale=10,  # AlphaEarth is 10m resolution
#                 geometries=True
#             )
#             print(f"SampleRegions configured for year {year}. Returning FeatureCollection.")
#         else:
#             print(f"No training points within mosaic bounds for year {year}, returning empty samples collection.")
#     else:
#         print(f"No images found for year {year} in date range, returning empty samples collection.")

#     print(f"--- Finished extractEmbeddings_mosaic_revised for year {year} ---")
#     return samples

# # 8. EXTRACT FOR MULTIPLE YEARS (using revised mosaic approach)
# print('--- Debugging Extraction for Multiple Years (Revised Mosaic Approach) ---')
# print('Extracting embeddings for multiple years...')
# embeddings2018_mosaic = extractEmbeddings_mosaic_revised(2018)
# embeddings2021_mosaic = extractEmbeddings_mosaic_revised(2021)  # Pre-China ban
# embeddings2024_mosaic = extractEmbeddings_mosaic_revised(2024)  # Post-ban

# print('Embeddings extraction configured successfully! Ready for export.')
# print('--- Finished Extraction for Multiple Years (Revised Mosaic Approach) ---')

--- Debugging Extraction for Multiple Years (Revised Mosaic Approach) ---
Extracting embeddings for multiple years...
--- Starting extractEmbeddings_mosaic_revised for year 2018 ---
Filtering AlphaEarth collection for date range: 2018-01-01 to 2019-01-01
Number of images for year 2018 in date range: 10635
Mosaic image created for year 2018.
Filtering training points by mosaic bounds for year 2018...
Number of training points within mosaic bounds for year 2018: 12124
Configuring sampleRegions for year 2018...
SampleRegions configured for year 2018. Returning FeatureCollection.
--- Finished extractEmbeddings_mosaic_revised for year 2018 ---
--- Starting extractEmbeddings_mosaic_revised for year 2021 ---
Filtering AlphaEarth collection for date range: 2021-01-01 to 2022-01-01
Number of images for year 2021 in date range: 10614
Mosaic image created for year 2021.
Filtering training points by mosaic bounds for year 2021...
Number of training points within mosaic bounds for year 2021: 12124


## Create cell for export configuration

### Subtask:
Create a cell for the export configuration using the sampled embeddings from the revised extraction method.


**Reasoning**:
Configure the export tasks for the sampled embeddings using the previously defined selectors.



In [22]:
# # 9. EXPORT FOR CLASSIFIER TRAINING (using mosaic approach)
# print('--- Debugging Export Configuration (Mosaic Approach) ---')

# # Export 2018 data to Google Drive
# task2018_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2018_mosaic,
#     description='AlphaEarth_Mining_Training_2018_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Export 2021 data to Google Drive
# task2021_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2021_mosaic,
#     description='AlphaEarth_Mining_Training_2021_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Export 2024 data to Google Drive
# task2024_mosaic = ee.batch.Export.table.toDrive(
#     collection=embeddings2024_mosaic,
#     description='AlphaEarth_Mining_Training_2024_Mosaic_Revised',
#     folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
#     fileFormat='CSV',
#     selectors=selectors
# )

# # Uncomment to start the export tasks
# # task2018_mosaic.start()
# # task2021_mosaic.start()
# # task2024_mosaic.start()
# # print('Export tasks started. Check your Google Drive and Earth Engine Tasks tab.')

# print('Export tasks configured. To start export, uncomment the task.start() lines and run again.')
# print(f'This will configure export for 2018, 2021, and 2024 samples with {len(band_names)} embedding bands.')
# print('--- Finished Export Configuration (Mosaic Approach) ---')

--- Debugging Export Configuration (Mosaic Approach) ---
Export tasks configured. To start export, uncomment the task.start() lines and run again.
This will configure export for 2018, 2021, and 2024 samples with 64 embedding bands.
--- Finished Export Configuration (Mosaic Approach) ---


## Update debugging output

### Subtask:
Update debugging output to reflect the revised embedding extraction process and the successful configuration of export tasks.


**Reasoning**:
Review the existing print statements related to embedding extraction and export configuration and update them to accurately reflect the revised mosaic approach and the deferred computation.



In [30]:
# 7. EXTRACT EMBEDDINGS AT TRAINING POINTS FOR SPECIFIC YEAR (Revised Mosaic approach)

def extractEmbeddings_mosaic_revised(year):
    """Extract AlphaEarth embeddings for a specific year using a mosaic,
       returning the FeatureCollection without immediate computation."""
    print(f"--- Starting embedding extraction configuration for year {year} (Revised Mosaic approach) ---")
    start_date = f"{year}-01-01"
    end_date = f"{year + 1}-01-01"
    print(f"Filtering AlphaEarth collection for date range: {start_date} to {end_date}")

    # Filter the collection by date
    yearImageCollection = alphaEarth.filterDate(start_date, end_date)
    collection_size = yearImageCollection.size().getInfo()
    print(f"Number of images found for year {year} in date range: {collection_size}")

    samples = ee.FeatureCollection([]) # Initialize empty collection

    if collection_size > 0:
        # Create a mosaic of the images for the year
        yearImage = yearImageCollection.mosaic()
        print(f"Mosaic image created for year {year}.")

        print(f"Filtering training points by mosaic bounds for year {year}...")
        # Filter training points to only include those within the bounds of the mosaic
        filteredTrainingPoints = trainingPoints.filterBounds(yearImage.geometry())
        # Getting the size here is acceptable as it's on the training points, not the sampled image data
        num_filtered_points = filteredTrainingPoints.size().getInfo()
        print(f"Number of training points within mosaic bounds for year {year}: {num_filtered_points}")


        if num_filtered_points > 0:
            print(f"Configuring sampleRegions operation for year {year}...")
            # Perform the sampling. This returns a FeatureCollection, but we avoid
            # calling getInfo() on it here, deferring computation.
            samples = yearImage.sampleRegions(
                collection=filteredTrainingPoints,
                scale=10,  # AlphaEarth is 10m resolution
                geometries=True
            )
            print(f"SampleRegions configured for year {year}. Returning FeatureCollection for later export.")
        else:
            print(f"No training points within mosaic bounds for year {year}, returning empty samples collection.")
    else:
        print(f"No images found for year {year} in date range, returning empty samples collection.")

    print(f"--- Finished embedding extraction configuration for year {year} ---")
    return samples

# 8. EXTRACT FOR MULTIPLE YEARS (using revised mosaic approach)
print('--- Configuring Embedding Extraction for Multiple Years (Revised Mosaic Approach) ---')
print('Configuring embedding extraction tasks...')
embeddings2018_mosaic = extractEmbeddings_mosaic_revised(2018)
embeddings2021_mosaic = extractEmbeddings_mosaic_revised(2021)  # Pre-China ban
embeddings2024_mosaic = extractEmbeddings_mosaic_revised(2024)  # Post-ban

print('Embedding extraction configurations created successfully! These are FeatureCollections ready for export.')
print('--- Finished Configuring Embedding Extraction for Multiple Years ---')

# Define band names and selectors for export (already in a separate cell)
# Generate band selectors for all 64 AlphaEarth layers
# band_names = ['A' + str(i).zfill(2) for i in range(64)]
# selectors = ['label', 'location', 'year'] + band_names
# print(f"Selectors for export: {selectors}")


# 9. EXPORT FOR CLASSIFIER TRAINING (using mosaic approach)
print('\n--- Configuring Export Tasks (Revised Mosaic Approach) ---')

# Export 2018 data to Google Drive
task2018_mosaic = ee.batch.Export.table.toDrive(
    collection=embeddings2018_mosaic,
    description='AlphaEarth_Mining_Training_2018_Mosaic_Revised',
    folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
    fileFormat='CSV',
    selectors=selectors
)
print('Export task configured for 2018 data.')

# Export 2021 data to Google Drive
task2021_mosaic = ee.batch.Export.table.toDrive(
    collection=embeddings2021_mosaic,
    description='AlphaEarth_Mining_Training_2021_Mosaic_Revised',
    folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
    fileFormat='CSV',
    selectors=selectors
)
print('Export task configured for 2021 data.')


# Export 2024 data to Google Drive
task2024_mosaic = ee.batch.Export.table.toDrive(
    collection=embeddings2024_mosaic,
    description='AlphaEarth_Mining_Training_2024_Mosaic_Revised',
    folder='AlphaEarth_Mining_Training', # Optional: specify a folder in Drive
    fileFormat='CSV',
    selectors=selectors
)
print('Export task configured for 2024 data.')


--- Configuring Embedding Extraction for Multiple Years (Revised Mosaic Approach) ---
Configuring embedding extraction tasks...
--- Starting embedding extraction configuration for year 2018 (Revised Mosaic approach) ---
Filtering AlphaEarth collection for date range: 2018-01-01 to 2019-01-01
Number of images found for year 2018 in date range: 10635
Mosaic image created for year 2018.
Filtering training points by mosaic bounds for year 2018...
Number of training points within mosaic bounds for year 2018: 12124
Configuring sampleRegions operation for year 2018...
SampleRegions configured for year 2018. Returning FeatureCollection for later export.
--- Finished embedding extraction configuration for year 2018 ---
--- Starting embedding extraction configuration for year 2021 (Revised Mosaic approach) ---
Filtering AlphaEarth collection for date range: 2021-01-01 to 2022-01-01
Number of images found for year 2021 in date range: 10614
Mosaic image created for year 2021.
Filtering training po

In [31]:
print('\nExport tasks configured successfully. To start export, uncomment the task.start() lines and run again.')
print(f'This will export the configured 2018, 2021, and 2024 samples with {len(band_names)} embedding bands when the tasks are started.')
print('--- Finished Configuring Export Tasks (Revised Mosaic Approach) ---')

# Uncomment to start the export tasks
task2018_mosaic.start()
task2021_mosaic.start()
task2024_mosaic.start()
print('Export tasks started. Check your Google Drive and Earth Engine Tasks tab.')

# # monitor the progress of the export tasks in the Earth Engine Tasks tab
# Submitted tasks
# AlphaEarth_Mining_Training_2024_Mosaic_Revised
# ID: MH5WWJSOVGEH5V5OMBMA54RC
# Phase: Completed
# Runtime: 2m (started 2025-10-11 17:22:47 -0500)
# Attempted 1 time
# Priority: 100 (default)
# Batch compute usage: 106724.1875 EECU-seconds
# 2m
# AlphaEarth_Mining_Training_2021_Mosaic_Revised
# ID: UV7HHKGEVSVR4XORYODI4CJL
# Phase: Completed
# Runtime: 3m (started 2025-10-11 17:22:36 -0500)
# Attempted 1 time
# Priority: 100 (default)
# Batch compute usage: 90087.0547 EECU-seconds
# 3m
# AlphaEarth_Mining_Training_2018_Mosaic_Revised
# ID: I6YEDMAHD223QEBU6MNOKG72
# Phase: Completed
# Runtime: 2m (started 2025-10-11 17:22:36 -0500)
# Attempted 1 time
# Priority: 100 (default)
# Batch compute usage: 106359.8047 EECU-seconds


Export tasks started. Check your Google Drive and Earth Engine Tasks tab.

Export tasks configured successfully. To start export, uncomment the task.start() lines and run again.
This will export the configured 2018, 2021, and 2024 samples with 64 embedding bands when the tasks are started.
--- Finished Configuring Export Tasks (Revised Mosaic Approach) ---


## Summary:

### Data Analysis Key Findings

*   The part of the algorithm that defines the selectors for the 64 AlphaEarth embedding bands ('A00' to 'A63') along with 'label', 'location', and 'year' was correctly identified and is functional.
*   The revised mosaic approach for embedding extraction successfully configures the `sampleRegions` operation and returns an Earth Engine `FeatureCollection` without triggering immediate computation, thus avoiding previous timeout issues.
*   Debugging output confirms the number of images found for each year and the number of training points within the mosaic bounds.
*   The export tasks for the sampled embeddings from 2018, 2021, and 2024 are successfully configured using the generated selectors and the returned `FeatureCollection` objects.
*   The revised process defers the heavy computation of sampling and exporting the data to the Earth Engine background tasks, which are initiated when `task.start()` is called.

### Insights or Next Steps

*   The revised method effectively overcomes the computation limit by deferring the sampling and export computations to Earth Engine's background processing.
*   The next crucial step is to uncomment the `task.start()` lines in the export configuration cell to initiate the export of the sampled AlphaEarth embeddings to Google Drive for subsequent use in classifier training.


In [32]:

# 10. VISUALIZE TRAINING POINTS ON MAP
print('--- Debugging Visualization ---')
# Create a comprehensive visualization
Map2 = geemap.Map(zoom=2)
print("Geemap map created.")

# Get AlphaEarth image for 2018 by creating a median composite over a date range
# Ensure an image is obtained, even if no image has a 'year' property set to 2018
ae2018_collection = alphaEarth.filterDate('2018-01-01', '2019-01-01')
print(f"Number of images in 2018 date range collection: {ae2018_collection.size().getInfo()}")
ae2018 = ae2018_collection.median() if ae2018_collection.size().getInfo() > 0 else None
print(f"Median composite image for 2018: {ae2018.getInfo() if ae2018 else 'None'}")


# Add AlphaEarth as RGB (using bands A01, A16, A09)
vis_params = {
    'min': -0.3,
    'max': 0.3,
    'bands': ['A01', 'A16', 'A09']
}
print(f"Visualization parameters: {vis_params}")

if ae2018:
    Map2.addLayer(ae2018, vis_params, 'AlphaEarth 2018 RGB', False)
    print("AlphaEarth 2018 RGB layer added.")
else:
    print("No AlphaEarth 2018 image available to add as a layer.")

Map2.addLayer(miningLocations, {'color': 'FF0000'}, 'Mining Locations (Positive)')
print("Mining Locations layer added.")
Map2.addLayer(negativeLocations, {'color': '0000FF'}, 'Negative Samples')
print("Negative Samples layer added.")

# Center on first mining location
if miningLocations.size().getInfo() > 0:
    Map2.centerObject(miningLocations.first(), 12)
    print("Map centered on first mining location.")
else:
    print("No mining locations to center the map on.")


print('Training Points Summary:')
print('  Total:', trainingPoints.size().getInfo())
print('  Positive (Mining):', miningLocations.size().getInfo())
print('  Negative (Non-mining):', negativeLocations.size().getInfo())
if ae2018:
    print('\nAlphaEarth Bands:', ae2018.bandNames().getInfo())
else:
    print('\nAlphaEarth image not available, band names cannot be retrieved.')
print('\nMap Legend:')
print('  Red points = Bitcoin mining locations')
print('  Blue points = Negative samples (non-mining)')

print('--- Finished Visualization ---')
Map2

--- Debugging Visualization ---
Geemap map created.
Number of images in 2018 date range collection: 10635
Median composite image for 2018: {'type': 'Image', 'bands': [{'id': 'A00', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A01', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A02', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A03', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A04', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A05', 'data_type': {'type': 'PixelType', 'precision': 'double'}, 'crs': 'EPSG:4326', 'crs_transform': [1, 0, 0, 0, 1, 0]}, {'id': 'A06', 'data_type': {'type': 'PixelType'

Map(center=[34.34817123, 62.19966888], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=…

In [36]:
from google.colab import drive
drive.mount('/content/drive/',force_remount=True)  # Change folder name here


Mounted at /content/drive/


In [33]:
%%bash
cd /content/drive/My\ Drive/AlphaEarthHack
ls -la data/training/

ls: cannot access 'data/training/': No such file or directory
